# Buscar perfiles más parecidos a los jugadores del club

## Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

## Rutas y variables

In [2]:
# Rutas de Archivos
RUTA_DATA = Path.cwd().parent.parent / 'data'
RUTA_MERCADO = RUTA_DATA / 'scout' / 'mercado_viable_2627.csv'
RUTA_UDLP = RUTA_DATA / 'udlaspalmas' / 'Individual Las Palmas 2526.xlsx'

# Diccionario
DICCIONARIO_COLUMNAS = {
    'Player': 'Jugador', 'Team': 'Equipo', 'Position': 'Posicion', 
    'Age': 'Edad', 'Foot': 'Pie', 'Pie': 'Pie',
    'Defensive duels per 90': 'Duelos_Defensivos_P90', 'Duelos defensivos/90': 'Duelos_Defensivos_P90',
    'Interceptions per 90': 'Interceptaciones_P90', 'Interceptaciones/90': 'Interceptaciones_P90',
    'Aerial duels per 90': 'Duelos_Aereos_P90', 'Duelos aéreos en los 90': 'Duelos_Aereos_P90',
    'Passes to final third per 90': 'Pases_Ultimo_Tercio_P90', 'Pases en el último tercio/90': 'Pases_Ultimo_Tercio_P90',
    'Progressive runs per 90': 'Conducciones_Progresivas_P90', 'Carreras en progresión/90': 'Conducciones_Progresivas_P90',
    'Key passes per 90': 'Pases_Clave_P90', 'Jugadas claves/90': 'Pases_Clave_P90',
    'Dribbles per 90': 'Regates_P90', 'Regates/90': 'Regates_P90',
    'Crosses per 90': 'Centros_P90', 'Centros/90': 'Centros_P90',
    'xA per 90': 'xA_P90', 'xA/90': 'xA_P90',
    'Goals per 90': 'Goles_P90', 'Goles/90': 'Goles_P90',
    'xG per 90': 'xG_P90', 'xG/90': 'xG_P90',
    'Touches in box per 90': 'Toques_Area_P90', 'Toques en el área de penalti/90': 'Toques_Area_P90',
    'Shots per 90': 'Remates_P90', 'Remates/90': 'Remates_P90',
    'Offensive duels per 90': 'Duelos_Ofensivos_P90', 'Duelos atacantes/90': 'Duelos_Ofensivos_P90'
}

# Vector de Comparación global
METRICAS_SIMILITUD = [
    'Duelos_Defensivos_P90', 'Interceptaciones_P90', 'Duelos_Aereos_P90',
    'Pases_Ultimo_Tercio_P90', 'Conducciones_Progresivas_P90', 'Pases_Clave_P90',
    'Regates_P90', 'Centros_P90', 'xA_P90', 'Goles_P90', 'xG_P90',
    'Toques_Area_P90', 'Remates_P90', 'Duelos_Ofensivos_P90'
]

# Familias de posición para ampliar la búsqueda de mercado.
GRUPOS_POSICION = [
    ('GK', 'Portero'),
    ('LB|RB|LWB|RWB|LI|RI', 'Lateral'),
    ('CB', 'Defensa central'),
    ('DMF|CMF', 'Mediocentro'),
    ('AMF', 'Mediocentro ofensivo'),
    ('LW|RW', 'Extremo'),
    ('CF', 'Delantero centro'),
]

## Carga de datos

In [3]:
def normalizar_pie(pie):
    """Unifica la lateralidad sin importar si el Excel viene en inglés o español"""
    pie_str = str(pie).lower().strip()
    if pie_str in ['right', 'derecho']: return 'diestro'
    if pie_str in ['left', 'izquierdo']: return 'zurdo'
    if pie_str in ['both', 'ambidiestro']: return 'ambidiestro'
    return 'desconocido'

# Cargar la base de datos de mercado exportada previamente
df_mercado = pd.read_csv(RUTA_MERCADO).fillna(0)
df_mercado['Pie_Normalizado'] = df_mercado['Pie'].apply(normalizar_pie)

# Cargar y traducir los datos de Las Palmas
df_udlp = pd.read_excel(RUTA_UDLP).fillna(0)
df_udlp = df_udlp.rename(columns=DICCIONARIO_COLUMNAS)
df_udlp['Pie_Normalizado'] = df_udlp['Pie'].apply(normalizar_pie)

print(f"Bases de datos listas. Universo de mercado: {len(df_mercado)} jugadores.")

Bases de datos listas. Universo de mercado: 4653 jugadores.


## Motor similitud

In [4]:
def obtener_patron_posicion(posicion_codigo, grupos=GRUPOS_POSICION):
    """
    Traduce un código de posición específico de Wyscout (ej. 'LCB', 'RAMF')
    a la familia de posición a la que pertenece (ej. 'Defensa central',
    'Mediocentro ofensivo'), para poder buscar clones en un grupo de
    mercado más amplio en lugar de exigir el código exacto.
    """
    for patron, _nombre in grupos:
        if re.search(patron, posicion_codigo, flags=re.IGNORECASE):
            return patron
    # Si no encaja en ningún grupo conocido, se usa el código tal cual
    return posicion_codigo

In [5]:
def buscar_clon_estadistico(nombre_jugador, df_equipo, df_global, metricas, estricto_pie=True):
    """
    Busca al jugador más similar en la base de datos global utilizando Similitud del Coseno.
    Filtra el mercado por familia de posición (no por código exacto de Wyscout) y,
    si procede, por pie dominante, relajando este último filtro si no deja candidatos.
    """
    # Aislar al jugador objetivo (Target)
    target = df_equipo[df_equipo['Jugador'].str.contains(nombre_jugador, case=False, na=False)]
    if target.empty:
        print(f"Error: No se ha encontrado a '{nombre_jugador}' en la plantilla de la UDLP.")
        return pd.DataFrame()

    # Extraemos sus metadatos vitales
    target = target.iloc[0]
    posiciones_target = [p.strip() for p in str(target['Posicion']).split(',')]  # Ej: ['LW', 'LAMF']
    posicion_principal = posiciones_target[0]  # Cogemos su rol principal para buscar clones
    pie_target = target['Pie_Normalizado']

    patron_posicion = obtener_patron_posicion(posicion_principal)

    print(f"Escaneando: {target['Jugador']}")
    print(f"Posición principal: {posicion_principal} (grupo: {patron_posicion}) | Pie Dominante: {pie_target.upper()}")

    # Filtrar el mercado por familia de posición
    mask_posicion = df_global['Posicion'].str.contains(patron_posicion, case=False, na=False, regex=True)
    df_filtrado = df_global[mask_posicion].copy()

    if df_filtrado.empty:
        print("No hay jugadores en el mercado viable que cumplan la familia de posición.")
        return pd.DataFrame()

    # Filtrar por pie, con fallback si el filtro estricto deja el mercado vacío
    if estricto_pie and pie_target != 'desconocido':
        df_mismo_pie = df_filtrado[df_filtrado['Pie_Normalizado'] == pie_target]
        if df_mismo_pie.empty:
            print(f"Aviso: no hay candidatos con pie '{pie_target}' en esta familia de posición; "
                  f"se amplía la búsqueda a todos los pies.")
        else:
            df_filtrado = df_mismo_pie

    # Preparar vectores matemáticos para el algoritmo
    df_modelo = pd.concat([pd.DataFrame([target]), df_filtrado]).reset_index(drop=True)

    # Validar que las métricas existen
    metricas_validas = [m for m in metricas if m in df_modelo.columns]
    X = df_modelo[metricas_validas].astype(float)

    # Escalar los datos (Z-Score)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Calcular Similitud del Coseno
    vector_target = X_scaled[0].reshape(1, -1)
    similitudes = cosine_similarity(vector_target, X_scaled)[0]

    # Mapear resultados
    df_filtrado['Porcentaje_Clonacion'] = (similitudes[1:] * 100).round(2)

    # Formatear la salida ejecutiva
    cols_visuales = ['Jugador', 'Equipo', 'Edad', 'Valor_MEur', 'Fin_Contrato', 'Carpeta_Origen', 'Porcentaje_Clonacion']
    cols_visuales = [c for c in cols_visuales if c in df_filtrado.columns]

    return df_filtrado.sort_values('Porcentaje_Clonacion', ascending=False)[cols_visuales].head(10).reset_index(drop=True)

## Aplicación del modelo

In [8]:
# Escribe aquí el nombre del jugador de la plantilla 25/26 que quieras buscar
JUGADOR_A_SUSTITUIR = "Kirian Rodríguez"

# Búsqueda
clones = buscar_clon_estadistico(
    nombre_jugador=JUGADOR_A_SUSTITUIR,
    df_equipo=df_udlp,
    df_global=df_mercado,
    metricas=METRICAS_SIMILITUD,
    estricto_pie=True
)

if not clones.empty:
    print(f"\nTop 10 parecidos:")
    display(clones)

Escaneando: Kirian Rodríguez
Posición principal: RCMF (grupo: DMF|CMF) | Pie Dominante: DIESTRO

Top 10 parecidos:


,Jugador,Equipo,Edad,Valor_MEur,Fin_Contrato,Carpeta_Origen,Porcentaje_Clonacion
0,T. Land,Groningen,20.0,0.00,2029-06-30,ligas,83.43
1,A. Varela,Porto,24.0,0.00,2030-06-30,ligas,65.08
2,J. Clasie,AZ,34.0,0.00,2028-06-30,ligas,64.93
3,P. Rosario,Porto,29.0,0.00,2029-06-30,ligas,61.74
4,Julen Huidobro,Barakaldo,30.0,0.15,2026-06-30,1rfef,55.12
5,J. Schouten,PSV,29.0,0.00,2030-06-30,ligas,54.07
6,J. Veerman,PSV,27.0,0.00,2028-06-30,ligas,53.51
7,João Moutinho,Sporting Braga,39.0,0.00,2026-06-30,ligas,52.81
8,M. Hjulmand,Sporting CP,26.0,0.00,2028-06-30,ligas,51.42
9,H. Morita,Sporting CP,31.0,0.00,2026-06-30,ligas,50.80


## Decisión final

In [9]:
if clones.empty:
    print("No se puede calcular la decisión final: la búsqueda de clones no devolvió candidatos.")
else:
    # Cargar el Top
    ruta_seleccion = Path.cwd().parent.parent / 'data' / 'seleccion' / 'top_opciones_mercado_2627.csv'
    df_opciones_sistema = pd.read_csv(ruta_seleccion)

    # Realizar copia del DF
    df_clones = clones.copy()

    # Fusionar algoritmos
    df_decision = pd.merge(
        df_opciones_sistema[['Jugador', 'Equipo', 'Valor_MEur', 'Score_Fit']],
        df_clones[['Jugador', 'Porcentaje_Clonacion']],
        on='Jugador',
        how='inner'
    )

    if df_decision.empty:
        print("Ningún candidato coincide entre el buscador de similitud y el top de opciones de mercado.")
    else:
        # Cálculo de score
        df_decision['Score_Global'] = ((df_decision['Score_Fit'] * 0.5) + (df_decision['Porcentaje_Clonacion'] * 0.5)).round(2)

        # Ordenar al ganador absoluto
        df_decision = df_decision.sort_values('Score_Global', ascending=False).reset_index(drop=True)

        print("Ranking fichajes:")
        display(df_decision)

        # Scatter plot
        plt.figure(figsize=(10, 8), facecolor='#F8FAFC')
        ax = plt.gca()
        ax.set_facecolor('#F8FAFC')

        # Crear el gráfico de dispersión
        sns.scatterplot(
            data=df_decision,
            x='Score_Fit',
            y='Porcentaje_Clonacion',
            size='Score_Global',
            sizes=(100, 500),
            color='#005BBB',
            edgecolor='#FFD700',
            linewidth=2,
            legend=False
        )

        # Añadir líneas de media para crear los 4 cuadrantes
        media_fit = df_decision['Score_Fit'].mean()
        media_clon = df_decision['Porcentaje_Clonacion'].mean()
        plt.axvline(media_fit, color='gray', linestyle='--', alpha=0.5)
        plt.axhline(media_clon, color='gray', linestyle='--', alpha=0.5)

        # Añadir el nombre de cada jugador al lado de su burbuja
        for i, row in df_decision.iterrows():
            plt.text(row['Score_Fit'] + 0.3, row['Porcentaje_Clonacion'] + 0.3,
                     row['Jugador'], fontsize=10, fontweight='bold', color='#1E293B')

        # Añadir etiquetas de los cuadrantes
        plt.text(df_decision['Score_Fit'].max(), df_decision['Porcentaje_Clonacion'].max(),
                 'Fichaje ideal', color='green', fontsize=12, fontweight='bold', ha='right')

        plt.title(f'Visualización del relevo de {JUGADOR_A_SUSTITUIR}', fontsize=16, fontweight='bold', color='#003057', pad=20)
        plt.xlabel('Ajuste al estilo Las Palmas', fontsize=12, fontweight='bold')
        plt.ylabel(f'Similitud con {JUGADOR_A_SUSTITUIR}', fontsize=12, fontweight='bold')

        plt.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        plt.show()

Ningún candidato coincide entre el buscador de similitud y el top de opciones de mercado.
